# Segmentation model score comparison

This notebook aggregates evaluation score CSV files generated by the three segmentation pipelines and produces comparison figures and tables.

It compares the following model types:

1. Per-species models
2. Mixed-species models
3. Domain-adaptive models

Outputs are saved under `evaluation_outputs/`.

In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

mpl.rcParams["pdf.fonttype"] = 42

# Run this notebook from the repository root.
# If the score folders are stored elsewhere, edit ROOT_DIR or SCORE_ROOTS below.
ROOT_DIR = Path(".").resolve()

OUTPUT_DIR = ROOT_DIR / "evaluation_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCORE_ROOTS = {
    "per_species": ROOT_DIR / "mrcnn_by_sp" / "inference_results_without_clearborder",
    "mixed_species": ROOT_DIR / "mrcnn_multisp" / "inference_results_without_clearborder",
    "domain_adaptive": ROOT_DIR / "mrcnn_multi_domain" / "inference_results_without_clearborder",
}

base_singlesp = SCORE_ROOTS["per_species"]
base_multisp = SCORE_ROOTS["mixed_species"]
base_da = SCORE_ROOTS["domain_adaptive"]

print("ROOT_DIR:", ROOT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
for name, path in SCORE_ROOTS.items():
    print(f"{name}: {path} | exists={path.exists()}")

## Input score files

This section defines the score CSV files used for comparison.

For per-species models, each species has its own score CSV.  
For mixed-species and domain-adaptive models, a shared score CSV contains rows for multiple images, so the target row is selected using the `filename` column where available.

In [ ]:
species_to_image = {
    "acapod": "acapod2_01_img.png",
    "alllit": "alllit2_01_img.png",
    "burspi": "burspi1_01_img.png",
    "euclon": "euclon1_01_img.png",
    "exostr": "exostr8_01_img.png",
    "hakdac": "hakdac2_01_img.png",
}

species_to_subtitle = {
    "acapod": "Acacia",
    "alllit": "Allocasuarina",
    "burspi": "Bursaria",
    "euclon": "Eucalyptus",
    "exostr": "Exocarpos",
    "hakdac": "Hakea",
}

models = {
    "PS-256": lambda sp, img: base_singlesp / "256_thin_singlespecies_700_epoch" / "scores" / img.replace(".png", "_evaluation_scores.csv"),
    "PS-512": lambda sp, img: base_singlesp / "512_thin_singlespecies_700_epoch" / "scores" / img.replace(".png", "_evaluation_scores.csv"),
    "MS-256": lambda sp, img: base_multisp / "256_thin_multispecies_700_epoch.tar" / "scores" / "evaluation_scores.csv",
    "MS-512": lambda sp, img: base_multisp / "512_thin_multispecies_700_epoch.tar" / "scores" / "evaluation_scores.csv",
    "DA-256": lambda sp, img: base_da / "256_thin_multidomain_700_epoch.tar" / "scores" / "evaluation_scores.csv",
    "DA-512": lambda sp, img: base_da / "512_thin_multidomain_700_epoch.tar" / "scores" / "evaluation_scores.csv",
}

model_order = ["PS-256", "PS-512", "MS-256", "MS-512", "DA-256", "DA-512"]

class_labels = {
    "vessel": "Vessel",
    "fiber": "Fibre",
    "parenchyma": "Axial P",
    "ray": "Ray P",
}

model_labels = {
    "PS-256": "PS-256",
    "PS-512": "PS-512",
    "MS-256": "MS-256",
    "MS-512": "MS-512",
    "DA-256": "DA-256",
    "DA-512": "DA-512",
}

## Helper functions

These functions load the score CSV files, select the correct image row, and extract requested metrics.

In [ ]:
def load_target_row(model_label, path, img_name):
    """Load the score row corresponding to the target image."""
    path = Path(path)
    df = pd.read_csv(path)

    img_base = img_name.replace(".png", "")

    if "filename" in df.columns:
        df_target = df[df["filename"] == img_base]

        if df_target.empty:
            print(f"[WARN] No matching row for {img_base} found in {model_label}: {path}")
            return None

        return df_target.iloc[0]

    # Per-species score files usually contain a single row.
    return df.iloc[0]


def get_metric_value(model_label, path_fn, species, img_name, metric_name):
    """Return one scalar metric value for one model-image pair."""
    path = Path(path_fn(species, img_name))

    if not path.exists():
        print(f"[ERROR] File not found: {path}")
        return None

    row = load_target_row(model_label, path, img_name)

    if row is None:
        return None

    if metric_name not in row.index:
        print(f"[WARN] {metric_name} not found in {path}")
        return None

    return row[metric_name]


def collect_score_rows():
    """Collect the selected score row for each species and model."""
    records = []

    for sp, img in species_to_image.items():
        for model_name, path_fn in models.items():
            path = Path(path_fn(sp, img))

            if not path.exists():
                print(f"[ERROR] File not found: {path}")
                continue

            row = load_target_row(model_name, path, img)

            if row is None:
                continue

            row_dict = row.to_dict()
            row_dict["image_name"] = img
            row_dict["species"] = sp
            row_dict["model"] = model_name

            records.append(row_dict)

    return pd.DataFrame(records)

## Aggregate all model scores

This section collects all available score rows into one table and saves it for reference.

In [ ]:
df_scores = collect_score_rows()

if df_scores.empty:
    raise FileNotFoundError(
        "No score files were loaded. Check ROOT_DIR, SCORE_ROOTS, and score CSV paths."
    )

cols = ["species", "image_name", "model"] + [
    c for c in df_scores.columns
    if c not in ["species", "image_name", "model"]
]

df_scores = df_scores[cols]

all_scores_path = OUTPUT_DIR / "all_model_scores.csv"
df_scores.to_csv(all_scores_path, index=False)

print(f"Saved: {all_scores_path}")

df_scores.head()

## Overall performance comparison

This section plots image-level and model-average scores for:

- mean Dice
- mean IoU
- accuracy

In [ ]:
def plot_model_scores(metric_name="mean_dice"):
    colors = [
        "lightcoral",
        "firebrick",
        "lightskyblue",
        "steelblue",
        "khaki",
        "goldenrod",
    ]

    all_scores = {}

    for species, img_name in species_to_image.items():
        scores = []

        for label, path_fn in models.items():
            value = get_metric_value(
                model_label=label,
                path_fn=path_fn,
                species=species,
                img_name=img_name,
                metric_name=metric_name,
            )

            scores.append(value)

        all_scores[img_name] = scores

    num_models = len(model_order)

    model_averages = []

    for i in range(num_models):
        valid_scores = [
            all_scores[img][i]
            for img in all_scores
            if all_scores[img][i] is not None
        ]

        model_averages.append(
            np.mean(valid_scores) if valid_scores else None
        )

    x = [i * 1.15 for i in range(len(all_scores))]
    x_avg = max(x) + 2

    custom_labels = [
        "Acacia",
        "Allocasuarina",
        "Bursaria",
        "Eucalyptus",
        "Exocarpos",
        "Hakea",
        "Model Average",
    ]

    width = 0.12

    fig, ax = plt.subplots(figsize=(14, 6))

    for i, (label, color) in enumerate(zip(model_order, colors)):
        offset = (i - num_models / 2) * width + width / 2

        model_scores = [
            all_scores[img][i]
            for img in all_scores
        ]

        ax.bar(
            [xi + offset for xi in x],
            model_scores,
            width,
            color=color,
            label=label,
        )

        if model_averages[i] is not None:
            ax.bar(
                x_avg + offset,
                model_averages[i],
                width,
                color=color,
            )

    ax.set_xticks(x + [x_avg])
    ax.set_xticklabels(custom_labels, rotation=45, ha="right")
    ax.set_ylabel(metric_name.replace("_", " ").title())
    ax.set_title(f"{metric_name.replace('_', ' ').title()} per Image for Each Model")
    ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5))

    plt.tight_layout()

    df_metric = pd.DataFrame.from_dict(
        all_scores,
        orient="index",
        columns=model_order,
    )

    df_metric.index.name = "Image"

    df_avg = pd.DataFrame(
        [model_averages],
        index=["Species Average"],
        columns=model_order,
    )

    df_all = pd.concat([df_metric, df_avg])

    csv_path = OUTPUT_DIR / f"{metric_name}_scores.csv"
    pdf_path = OUTPUT_DIR / f"{metric_name}.pdf"
    png_path = OUTPUT_DIR / f"{metric_name}.png"

    df_all.to_csv(csv_path)
    plt.savefig(pdf_path, bbox_inches="tight")
    plt.savefig(png_path, dpi=300, bbox_inches="tight")

    print(f"Saved: {csv_path}")
    print(f"Saved: {pdf_path}")
    print(f"Saved: {png_path}")

    plt.show()

In [ ]:
plot_model_scores(metric_name="mean_dice")
plot_model_scores(metric_name="mean_iou")
plot_model_scores(metric_name="accuracy")

## Save metric tables

This section saves one table per overall metric.  
Rows correspond to target images and columns correspond to model variants.

In [ ]:
metrics = ["mean_dice", "mean_iou", "accuracy"]

for metric_name in metrics:
    results_df = pd.DataFrame(index=species_to_image.values())

    for model_name, path_fn in models.items():
        model_scores = []

        for species, img in species_to_image.items():
            value = get_metric_value(
                model_label=model_name,
                path_fn=path_fn,
                species=species,
                img_name=img,
                metric_name=metric_name,
            )

            model_scores.append(value)

        results_df[model_name] = model_scores

    csv_path = OUTPUT_DIR / f"{metric_name}_table.csv"
    results_df.to_csv(csv_path)

    print(f"Saved: {csv_path}")

## Per-class score heatmaps

This section plots per-class Dice and IoU scores for each image.

Rows correspond to cell classes and columns correspond to model variants.

In [ ]:
def plot_heatmap_per_image_custom(
    models,
    species_to_image,
    score_type="dice",
    class_order=None,
    class_labels=class_labels,
    model_labels=model_labels,
    title_prefix=None,
    output_name=None,
):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    last_idx = -1

    for idx, (sp, img_name) in enumerate(species_to_image.items()):
        last_idx = idx
        records = []

        for model_label, path_fn in models.items():
            path = Path(path_fn(sp, img_name))

            if not path.exists():
                print(f"[ERROR] File not found: {path}")
                continue

            row = load_target_row(model_label, path, img_name)

            if row is None:
                continue

            df = pd.read_csv(path)

            for col in df.columns:
                if col.endswith(f"_{score_type}") and not col.startswith("mean_"):
                    cls = col.replace(f"_{score_type}", "")

                    if cls:
                        records.append({
                            "Class": cls,
                            "Model": model_label,
                            "Score": row[col],
                        })

        df_plot = pd.DataFrame(records)

        if df_plot.empty:
            print(f"No valid data found for {img_name}")
            axes[idx].axis("off")
            continue

        df_plot["Model"] = pd.Categorical(
            df_plot["Model"],
            categories=model_order,
            ordered=True,
        )

        if class_order:
            df_plot["Class"] = pd.Categorical(
                df_plot["Class"].map(class_labels).fillna(df_plot["Class"]),
                categories=[class_labels.get(c, c) for c in class_order],
                ordered=True,
            )

        df_plot["Model"] = df_plot["Model"].map(model_labels).fillna(df_plot["Model"])

        df_pivot = df_plot.pivot(
            index="Class",
            columns="Model",
            values="Score",
        )

        sns.heatmap(
            df_pivot,
            annot=True,
            fmt=".2f",
            cmap="coolwarm",
            vmin=0,
            vmax=1,
            ax=axes[idx],
            cbar=False,
        )

        axes[idx].set_title(
            f"{species_to_subtitle.get(sp, sp)} - {score_type.upper()}"
        )

    for j in range(last_idx + 1, len(axes)):
        axes[j].axis("off")

    if title_prefix:
        fig.suptitle(title_prefix, fontsize=16)
    else:
        fig.suptitle(
            f"{score_type.upper()} Score per Class for Each Image",
            fontsize=16,
        )

    plt.tight_layout(rect=[0, 0, 1, 0.95])

    if output_name is not None:
        pdf_path = OUTPUT_DIR / f"{output_name}.pdf"
        png_path = OUTPUT_DIR / f"{output_name}.png"

        plt.gcf().savefig(pdf_path, bbox_inches="tight")
        plt.gcf().savefig(png_path, dpi=300, bbox_inches="tight")

        print(f"Saved: {pdf_path}")
        print(f"Saved: {png_path}")

    plt.show()
    plt.close()

In [ ]:
narrow_keys = ["PS-256", "PS-512", "MS-256", "MS-512", "DA-256", "DA-512"]
narrow_models = {k: models[k] for k in narrow_keys}

plot_heatmap_per_image_custom(
    models=narrow_models,
    species_to_image=species_to_image,
    score_type="dice",
    class_order=["vessel", "fiber", "parenchyma", "ray"],
    class_labels=class_labels,
    model_labels=model_labels,
    title_prefix="Per-Class Dice Score for Each Image",
    output_name="per_class_dice_heatmap",
)

In [ ]:
plot_heatmap_per_image_custom(
    models=narrow_models,
    species_to_image=species_to_image,
    score_type="iou",
    class_order=["vessel", "fiber", "parenchyma", "ray"],
    class_labels=class_labels,
    model_labels=model_labels,
    title_prefix="Per-Class IoU Score for Each Image",
    output_name="per_class_iou_heatmap",
)

## Final outputs

This notebook saves the following main outputs under `evaluation_outputs/`:

- `all_model_scores.csv`
- `mean_dice_scores.csv`
- `mean_iou_scores.csv`
- `accuracy_scores.csv`
- `mean_dice_table.csv`
- `mean_iou_table.csv`
- `accuracy_table.csv`
- `mean_dice.pdf`
- `mean_dice.png`
- `mean_iou.pdf`
- `mean_iou.png`
- `accuracy.pdf`
- `accuracy.png`
- `per_class_dice_heatmap.pdf`
- `per_class_dice_heatmap.png`
- `per_class_iou_heatmap.pdf`
- `per_class_iou_heatmap.png`